# V30 — Annual-state HMM and nested household-class model

**Coauthor review version — final 38-column Parquet input.**
Run the cells in order after setting the path. Dependencies: `numpy pandas scipy duckdb pyarrow tqdm`.
Install missing dependencies in a separate cell using `%pip install numpy pandas scipy duckdb pyarrow tqdm`, then restart the kernel.

## Agreed specification
For household $i$, year $t$, trip $j$, category $c$, state $s\in\{1,2\}$:

- State-independent incidence: $p_{cijt}=\operatorname{logit}^{-1}(a_c+b_c\,food\_spend_{ijt}/10)$.
- Choice gate $k$: $g_{kits}=\operatorname{logit}^{-1}(\alpha_k+\delta_k s+X_{it}'\beta_k)$.
  $X$ contains children under 18, income in $10,000, household size, and age in years.
- Cereal uses sequential gates: Low $=g_1$, Medium $=(1-g_1)g_2$, High $=(1-g_1)(1-g_2)$, conditional on a classified purchase.
- Five other categories have one binary choice gate each. Unconditional positive-outcome probabilities multiply the choice probability by incidence; zero has probability $1-p_c$.
- The joint emission multiplies six category probabilities, conditional on state and covariates. Annual log emissions sum all trip log emissions.
- Proposed transitions: $P(S_{it}=2\mid S_{i,t-1}=r)=\operatorname{logit}^{-1}(Z_{it}'\theta_r)$, $r=1,2$.
  $Z$ has an intercept and the six agreed current-year events. Unemployment is excluded.
- Nested model: $P=I_2$; a household retains one class for its full observed history. Choice coefficients are re-estimated.
- **Fixed common initial probability:** $\pi=P(S_{i,t_0}=2)$ equals the pooled 2007 fresh-FV share. It weights the first state for every household, including later entrants. It is never updated, counted as a free parameter, or assigned an SE. The 2007 observations have no separate likelihood contribution.

**Parameter counts:** incidence 12 + choice 42 + transitions 14 = **68 proposed**; **54 nested**.
State 1/2 are intended as Unhealthy/Healthy; inspect estimated profiles before interpreting these labels. The fixed asymmetric prior means labels cannot be swapped without also changing that prior.

## Sample and inference conventions
The default preserves V29's requirement of at least **three observed years per household**; set `MIN_OBSERVED_YEARS_PER_HOUSEHOLD=1` to retain all households. Input and retained counts are exported. The default also preserves V29's **one transition per successive observed year** convention, including gaps. Gaps are counted and explicitly reported; choose `YEAR_GAP_POLICY='error'` to require consecutive calendar years. This is an observed-year chain when gaps exist, not a transition for every unobserved calendar year.

Coefficient inference uses the inherited **block M-step Hessians**, holding posterior weights fixed, not the full observed-data HMM information matrix. Cross-block covariance is unavailable; latent-state uncertainty is omitted. Singular blocks receive blank inference. Posterior purchase profiles are weighted empirical frequencies, with household-cluster delta-method SEs treating posterior weights as fixed. Stars test the State 2 minus State 1 difference. These approximations can understate uncertainty.

This notebook uses the full retained 2008–2017 sample; it does not introduce a train/test split.


## Validation performed before delivery
The synthetic Parquet workflow completed for both models (120 households, 480 household-years, 5,760 trips). These are software checks, not empirical findings. Independent tests verified:
- Annual sufficient-statistic emissions match direct trip-by-trip evaluation.
- Forward likelihood matches explicit enumeration of every state path in four-year sequences.
- No-switch likelihood matches the exact two-class household mixture.
- The fixed initial probability stays unchanged; coefficient counts are 68 and 54.
- All synthetic coefficient SEs are finite; posterior state probabilities and conditional compositions normalize correctly.
- Accepted EM likelihoods are nondecreasing; all result files export successfully.
The confidential full dataset was unavailable here. Full-data runtime, convergence and Hessian rank must be assessed after running locally.


In [ ]:
import os

# Numerical-library thread settings MUST run before importing NumPy/SciPy.
# HMM_NUM_THREADS overrides the conservative default of min(4, detected CPUs).
_thread_text = os.environ.get("HMM_NUM_THREADS")
if _thread_text is None:
    HMM_NUM_THREADS = min(4, os.cpu_count() or 1)
else:
    try:
        HMM_NUM_THREADS = int(_thread_text)
    except (TypeError, ValueError) as exc:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.") from exc
    if HMM_NUM_THREADS < 1:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.")
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = str(HMM_NUM_THREADS)
print(f"Requested numerical threads = {HMM_NUM_THREADS}")
print("Thread settings take effect only before NumPy/SciPy import; after changing them, Restart Kernel and Run All.")

import gc
import math
import time
import warnings
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from scipy.stats import norm
try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        """Minimal progress fallback used only when tqdm is unavailable."""
        def __init__(self, iterable=None, total=None, desc="", **kwargs):
            self.iterable = iterable; self.total = total; self.desc = desc; self.n = 0
        def __enter__(self):
            print(self.desc); return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def __iter__(self):
            for item in self.iterable:
                yield item; self.n += 1
        def update(self, value=1):
            self.n += value
        def set_description_str(self, text):
            self.desc = text; print(text)
        def set_postfix_str(self, text):
            pass
        def set_postfix(self, **kwargs):
            pass


import duckdb
import json
INPUT_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel.parquet")
OUTPUT_DIR = INPUT_PATH.parent / 'V30_Final_HMM_Results'
RUN_MODE = 'full_estimation'  # 'full_estimation', 'validate_only', or 'synthetic'
MAX_EM_ITERS = 150
SYNTHETIC_EM_ITERS = 40
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 3
YEAR_GAP_POLICY = 'observed_step'  # 'error' rejects calendar-year gaps.
DUCKDB_MEMORY_LIMIT = '4GB'
CHECK_DUPLICATE_TRIPS = True
TOL=1e-3
EM_REL_TOL=1e-10
EM_STABLE_ITERS=3
EM_PRINT_EVERY=5
OPT_FTOL=1e-14
OPT_GTOL=1e-8
OPT_NEWTON_STEPS=20
SEED=20261030
CEREAL_CHOICE_WEIGHT=1.0
FRESH_SHARE_2007_COL='initial_fresh_fv_share_2007'
TRANSITION_EVENT_COLUMNS=['event_new_child','getting_married','getting_divorce_or_separation','moving','getting_job','retirement']
TRANSITION_X_COLS=['intercept',*TRANSITION_EVENT_COLUMNS]
GATE_COVARIATE_COLS=['K_1','income10k_imputed','family_size_numeric','age']
GATE_X_COLS=['intercept','state_num',*GATE_COVARIATE_COLS]
CATEGORY_CONFIG=OrderedDict()
_spec=[
 ('fv',['fresh','nonfresh'],['fresh_spending','non_fresh_spending']),
 ('milk',['skimmed','other_milk'],['skimmed_milk_spending','low_fat_milk_spending','other_milk_spending']),
 ('yogurt',['fat_free','other_yogurt'],['skimmed_yogurt_spending','low_fat_yogurt_spending','other_yogurt_spending']),
 ('cereal',['low_sugar','medium_sugar','high_sugar'],['low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending']),
 ('snack',['healthy','unhealthy'],['healthy_snack_spending','unhealthy_snack_spending']),
 ('beverage',['healthier','less_healthy'],['healthy_beverage_spending','unhealthy_beverage_spending'])]
for c,labels,spending in _spec:
 gates=[dict(key=c+'_'+labels[0],prefix=c+'_'+labels[0]+'_gate',eligible_codes=tuple(range(1,len(labels)+1)),target_code=1)]
 if c=='cereal': gates.append(dict(key='cereal_medium_given_notlow',prefix='cereal_medium_given_notlow_gate',eligible_codes=(2,3),target_code=2))
 CATEGORY_CONFIG[c]=dict(y_col=c+'_y',outcomes=OrderedDict(enumerate(['no_classified_purchase' if c=='cereal' else 'no_purchase',*labels])),gates=gates,spending=spending,choice_weight=1.0)
CATEGORY_NAMES=list(CATEGORY_CONFIG)
GATE_KEYS=[g['key'] for c in CATEGORY_CONFIG.values() for g in c['gates']]
np.random.seed(SEED)


## 1. Numerical engine
The V29 analytic gradients, curvature scaling, warm starts, Newton refinement, and downhill-step rejection are retained. No statistical ridge penalty is introduced. Only small annual arrays enter EM.

In [ ]:
@dataclass
class GateBlock:
    """Annual outcome counts mapped to one shared unique-covariate design."""

    key: str
    category: str
    prefix: str
    x_base: np.ndarray
    target: np.ndarray
    hy_pos: np.ndarray
    eligible_rows: int
    choice_weight: float
    success_count: np.ndarray
    failure_count: np.ndarray
    design_pos: np.ndarray


@dataclass
class ModelArrays:
    """NumPy model data with cached household-year and gate structures."""

    purchase: Dict[str, np.ndarray]
    outcome: Dict[str, np.ndarray]
    spend: Dict[str, np.ndarray]
    food_total_spend: np.ndarray
    food_total_spend_10: np.ndarray
    gate_covariates: Dict[str, np.ndarray]
    gate_blocks: Dict[str, GateBlock]
    trip_hy_pos: np.ndarray
    household_year: pd.DataFrame
    hy_key: pd.MultiIndex
    hh_blocks: Dict[str, Dict[str, np.ndarray]]
    households: List[str]
    years_by_household: Dict[str, List[int]]
    hh_starts: np.ndarray
    hh_ends: np.ndarray
    transition_x: np.ndarray
    transition_curr_pos: np.ndarray
    transition_row_of_hy: np.ndarray


@dataclass
class PreparedData:
    """Annual sufficient statistics and model buffers.

    With RETAIN_TRIP_DATA=False, df is a count-only RangeIndex shell carrying
    the trip count and sorting audit. Raw trip buffers are released after the
    incidence fits; all subsequent EM operations use annual/design counts.
    """

    df: pd.DataFrame
    hy: pd.DataFrame
    arrays: ModelArrays
    annual_stats: pd.DataFrame


@dataclass
class EstimationResult:
    """Fitted model blocks and final optimizer diagnostics."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    beta_from1: np.ndarray
    beta_from2: np.ndarray
    pi_state2: float
    gamma_df: pd.DataFrame
    history_df: pd.DataFrame
    initial_state_table: pd.DataFrame
    transition_table: pd.DataFrame
    log_likelihood: float
    transition_fits: Optional[Dict[str, Dict[str, Any]]] = None
    optimizer_diagnostics_df: Optional[pd.DataFrame] = None


def safe_log(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Take a stable logarithm of probabilities."""
    return np.log(np.clip(x, eps, 1.0))


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Stable logistic transform."""
    return expit(x)


def significance_stars(p_value: float) -> str:
    """Return conventional significance stars."""
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


def ensure_columns(df: pd.DataFrame, required_cols: Sequence[str]) -> None:
    """Raise a clear error when required columns are missing."""
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def first_nonmissing(series: pd.Series) -> float:
    """Return the first observed value, preserving missingness when all values are missing."""
    observed = series.dropna()
    return observed.iloc[0] if len(observed) else np.nan


def _optimizer_diagnostics(
    result: Any,
    block_name: str,
    objective_value: float,
    gradient: np.ndarray,
    n_observations: int,
    effective_weight: float,
    covariance_success: bool,
    solver: str,
) -> Dict[str, Any]:
    return {
        "scaled_gradient_inf": float(getattr(result, 'scaled_gradient_inf', np.nan)),
        "initial_objective": float(getattr(result, 'initial_objective', np.nan)),
        "objective_improvement": float(getattr(result, 'objective_improvement', np.nan)),
        "reverted_to_start": bool(getattr(result, 'reverted_to_start', False)),
        "lbfgs_message": str(getattr(result, 'lbfgs_message', '')),
        "newton_refinements": int(getattr(result, 'newton_refinements', 0)),
        "model_block": block_name,
        "solver": solver,
        "optimizer_success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "iterations": int(getattr(result, "nit", -1)),
        "function_evaluations": int(getattr(result, "nfev", -1)),
        "gradient_evaluations": int(getattr(result, "njev", -1)),
        "final_objective": float(objective_value),
        "gradient_norm": float(np.linalg.norm(gradient)),
        "n_observations": int(n_observations),
        "effective_weight": float(effective_weight),
        "covariance_hessian_success": bool(covariance_success),
    }


def _finish_logit_fit(
    result: Any,
    hessian_function: Any,
    gradient_function: Any,
    param_names: Sequence[str],
    block_name: str,
    objective_function: Any,
    n_observations: int,
    effective_weight: float,
    compute_se: bool,
    solver: str,
) -> Dict[str, Any]:
    beta = np.asarray(result.x, dtype=np.float64)
    covariance_success = False
    if compute_se:
        try:
            hessian = np.asarray(hessian_function(beta), dtype=np.float64)
            covariance = np.linalg.pinv(hessian)
            standard_error = np.sqrt(np.clip(np.diag(covariance), 0.0, np.inf))
            covariance_success = bool(np.isfinite(covariance).all())
        except Exception as exc:
            warnings.warn(f"{block_name}: Hessian/covariance failed: {exc}", RuntimeWarning)
            hessian = np.full((len(beta), len(beta)), np.nan)
            covariance = np.full((len(beta), len(beta)), np.nan)
            standard_error = np.full(len(beta), np.nan)
    else:
        hessian = np.full((len(beta), len(beta)), np.nan)
        covariance = np.full((len(beta), len(beta)), np.nan)
        standard_error = np.full(len(beta), np.nan)

    gradient = np.asarray(gradient_function(beta), dtype=np.float64)
    objective_value = float(objective_function(beta))
    with np.errstate(divide="ignore", invalid="ignore"):
        z_value = beta / standard_error
        p_value = 2.0 * (1.0 - norm.cdf(np.abs(z_value)))
    table = pd.DataFrame({
        "parameter": list(param_names),
        "estimate": beta,
        "std_error": standard_error,
        "z_value": z_value,
        "p_value": p_value,
    })
    table["significance_stars"] = table["p_value"].map(significance_stars)
    diagnostics = _optimizer_diagnostics(
        result, block_name, objective_value, gradient, n_observations,
        effective_weight, covariance_success, solver,
    )
    if not result.success:
        warnings.warn(
            f"Optimizer did not report success for {block_name}: "
            f"status={result.status}; message={result.message}",
            RuntimeWarning,
        )
    return {
        "_hessian_function": hessian_function,
        "result": result,
        "coef": beta,
        "hessian": hessian,
        "cov": covariance,
        "table": table,
        "nll": objective_value,
        "diagnostics": diagnostics,
    }


def _cached_value_gradient(callback):
    """Reuse one matrix product/sigmoid pass for both L-BFGS-B callbacks."""
    last_beta = None
    last_result = None
    def evaluate(beta):
        nonlocal last_beta, last_result
        if last_beta is None or not np.array_equal(beta, last_beta):
            last_beta = np.array(beta, copy=True)
            last_result = callback(beta)
        return last_result
    return evaluate


def _minimize_logit(evaluate, hess, initial, effective_weight, maxiter):
    """Scaled L-BFGS with gradient verification and damped Newton refinement.

    Scaling changes only optimizer coordinates, never data or likelihood.
    The starting coefficients are always a candidate, so a block M-step
    cannot decrease its expected complete-data log likelihood.
    """
    from scipy.optimize import OptimizeResult
    weight = max(float(effective_weight), 1.0)
    initial = np.array(initial, dtype=float, copy=True)
    curvature = np.maximum(np.diag(hess(np.zeros_like(initial))) / weight, 0.0)
    scale = np.sqrt(curvature)
    scale[scale < 1e-8] = 1.0
    calls = 0

    def objective(theta):
        nonlocal calls
        calls += 1
        value, gradient = evaluate(theta / scale)
        return value / weight, gradient / (weight * scale)

    theta0 = initial * scale
    f0, g0 = objective(theta0)
    best_theta, best_f = theta0.copy(), f0
    raw_message = 'Initial coefficients evaluated without optimization'
    nit = 0
    if maxiter > 0:
        res = minimize(objective, theta0, jac=True, method='L-BFGS-B', options={
            'maxiter': maxiter, 'maxfun': max(20000, 30 * maxiter),
            'ftol': OPT_FTOL, 'gtol': OPT_GTOL, 'maxls': 50, 'maxcor': 20,
        })
        raw_message, nit = str(res.message), int(res.nit)
        candidate_f, _ = objective(res.x)
        if np.isfinite(candidate_f) and candidate_f <= best_f:
            best_theta, best_f = res.x.copy(), candidate_f

    # A relative-function stop alone is insufficient on a massive likelihood.
    # Small dense Hessians (one logit block) make Newton polishing affordable.
    refinements = 0
    for _ in range(OPT_NEWTON_STEPS if maxiter > 0 else 0):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter * np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            direction, slope = -grad, -float(grad @ grad)
        accepted = False
        for backtrack in range(40):
            step = 0.5 ** backtrack
            theta = best_theta + step * direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4 * step * slope:
                best_theta, best_f = theta, candidate_f
                accepted = True
                refinements += 1
                break
        if not accepted:
            break
    # Recheck unscaled loss, preventing a rounding artifact in division from
    # accepting an objectively worse solution than the warm start.
    beta = best_theta / scale
    initial_value, _ = evaluate(initial)
    final_value, final_gradient = evaluate(beta)
    reverted = not np.isfinite(final_value) or final_value > initial_value
    if reverted:
        beta = initial
        final_value, final_gradient = evaluate(beta)
    scaled_gradient = final_gradient / (weight * scale)
    score = float(np.linalg.norm(scaled_gradient, ord=np.inf))
    success = bool(np.isfinite(score) and score <= OPT_GTOL)
    return OptimizeResult(
        x=beta, fun=final_value, jac=final_gradient, success=success,
        status=0 if success else 2, nit=nit + refinements, nfev=calls, njev=calls,
        message=('Scaled gradient tolerance met' if success else
                 'Scaled gradient tolerance NOT met; inspect this block'),
        scaled_gradient_inf=score, initial_objective=initial_value,
        objective_improvement=initial_value-final_value, reverted_to_start=reverted,
        lbfgs_message=raw_message, newton_refinements=refinements,
    )




def fit_weighted_logit(
    X, y, sample_weight=None, start=None, param_names=None, maxiter=300,
    compute_se=False, block_name="weighted_logit",
):
    """Weighted Bernoulli likelihood, evaluated once per optimizer candidate."""
    y = np.asarray(y, dtype=np.float64)
    weight = np.ones_like(y) if sample_weight is None else np.asarray(sample_weight, dtype=np.float64)
    if np.any(np.isfinite(weight) & (weight < 0)):
        raise ValueError(f"{block_name}: negative weights are not allowed.")
    names = list(param_names) if param_names is not None else [f"b{i}" for i in range(X.shape[1])]
    return fit_binomial_count_logit(
        X, weight*y, weight*(1-y), start, names, block_name, maxiter, compute_se,
    )


def fit_compressed_gate_logit(
    X_base, y, weight_state1, weight_state2, start, param_names, block_name,
    maxiter=300, compute_se=False,
):
    """Compatibility wrapper for the original two-state weighted-trip API."""
    y = np.asarray(y, dtype=np.float64)
    w1 = np.asarray(weight_state1, dtype=np.float64)
    w2 = np.asarray(weight_state2, dtype=np.float64)
    return fit_gate_count_logit(
        X_base, w1*y, w1*(1-y), w2*y, w2*(1-y), start,
        param_names, block_name, maxiter, compute_se,
    )


def fit_gate_count_logit(
    X_base, success1, failure1, success2, failure2, start,
    param_names, block_name, maxiter=300, compute_se=False,
):
    """Exact two-state binomial likelihood on unique, unrounded covariates.

    Counts already include annual posterior weights and the category weight.
    No binomial combinatorial constant is added (the original trip likelihood).
    """
    X_base = np.asarray(X_base, dtype=np.float64)
    counts = np.column_stack([success1, failure1, success2, failure2]).astype(np.float64, copy=False)
    if np.any(np.isfinite(counts) & (counts < 0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    keep = np.isfinite(counts).all(axis=1) & (counts.sum(axis=1)>0) & np.isfinite(X_base).all(axis=1)
    if not keep.all():
        X_base, counts = X_base[keep], counts[keep]
    if not len(counts):
        raise ValueError(f"{block_name}: no valid observations remain.")
    s1, f1, s2, f2 = counts.T
    t1, t2 = s1+f1, s2+f2
    p_count = X_base.shape[1]+1
    base_positions = np.r_[0, np.arange(2, p_count)]
    initial = np.zeros(p_count) if start is None else np.asarray(start, dtype=np.float64)

    def components(beta):
        shared = X_base @ beta[base_positions]
        return sigmoid(shared+beta[1]), sigmoid(shared+2*beta[1])

    def value_gradient(beta):
        p1, p2 = components(beta)
        shared = X_base @ beta[base_positions]
        eta1, eta2 = shared + beta[1], shared + 2 * beta[1]
        # Stable Bernoulli loss, exactly consistent with the analytic gradient.
        value = np.sum(s1*np.logaddexp(0, -eta1) + f1*np.logaddexp(0, eta1)
                       + s2*np.logaddexp(0, -eta2) + f2*np.logaddexp(0, eta2))
        r1, r2 = t1*p1-s1, t2*p2-s2
        out = np.empty(p_count)
        out[base_positions] = X_base.T @ (r1+r2)
        out[1] = np.sum(r1+2*r2)
        return float(value), out
    evaluate = _cached_value_gradient(value_gradient)

    def hess(beta):
        p1, p2 = components(beta)
        c1, c2 = t1*p1*(1-p1), t2*p2*(1-p2)
        out = np.empty((p_count, p_count))
        out[np.ix_(base_positions, base_positions)] = X_base.T @ (X_base*(c1+c2)[:, None])
        cross = X_base.T @ (c1+2*c2)
        out[base_positions, 1] = cross
        out[1, base_positions] = cross
        out[1, 1] = np.sum(c1+4*c2)
        return out

    result = _minimize_logit(evaluate, hess, initial, float(counts.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(counts), float(counts.sum()), compute_se,
        "scaled_lbfgs_newton_gate_logit",
    )


def fit_binomial_count_logit(
    X, success, failure, start, param_names, block_name, maxiter=300, compute_se=False,
):
    """Original binomial-count objective with a joint loss/gradient callback."""
    X = np.asarray(X, dtype=np.float64)
    success, failure = np.asarray(success, dtype=np.float64), np.asarray(failure, dtype=np.float64)
    if np.any(np.isfinite(success) & (success<0)) or np.any(np.isfinite(failure) & (failure<0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    total = success+failure
    keep = np.isfinite(success) & np.isfinite(failure) & (total>0) & np.isfinite(X).all(axis=1)
    if not keep.all():
        X, success, failure, total = X[keep], success[keep], failure[keep], total[keep]
    if not len(total):
        raise ValueError(f"{block_name}: no positive expected counts remain.")
    initial = np.zeros(X.shape[1]) if start is None else np.asarray(start, dtype=np.float64)

    def value_gradient(beta):
        eta = X @ beta
        p = sigmoid(eta)
        value = np.sum(success*np.logaddexp(0, -eta) + failure*np.logaddexp(0, eta))
        return float(value), X.T @ (total*p-success)
    evaluate = _cached_value_gradient(value_gradient)
    def hess(beta):
        p = sigmoid(X @ beta)
        return X.T @ (X*(total*p*(1-p))[:, None])

    result = _minimize_logit(evaluate, hess, initial, float(total.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(total), float(total.sum()), compute_se,
        "scaled_lbfgs_newton_binomial_logit",
    )


def finalize_covariance(fit: Dict[str, Any]) -> None:
    """Evaluate a final M-step Hessian once, with its original E-step weights."""
    callback = fit.pop('_hessian_function')
    hessian = np.asarray(callback(fit['coef']), dtype=np.float64)
    if not np.isfinite(hessian).all():
        raise ValueError('Nonfinite final block Hessian.')
    covariance = np.linalg.pinv(hessian)  # Same V29 pseudo-inverse convention.
    se = np.sqrt(np.clip(np.diag(covariance), 0.0, np.inf))
    with np.errstate(divide='ignore', invalid='ignore'):
        z = fit['coef'] / se
    table = fit['table']
    table['std_error'] = se
    table['z_value'] = z
    table['p_value'] = 2 * norm.sf(np.abs(z))
    table['significance_stars'] = table['p_value'].map(significance_stars)
    fit['hessian'], fit['cov'] = hessian, covariance
    fit['diagnostics']['covariance_hessian_success'] = bool(np.isfinite(covariance).all())
    fit['diagnostics']['hessian_rank'] = int(np.linalg.matrix_rank(hessian))
    fit['diagnostics']['parameter_count'] = len(se)


## 2. Parquet validation and exact sufficient statistics
DuckDB scans only required columns and can spill aggregation to disk. It builds one row per household-year plus one incidence row per distinct food-spending value. Trips are never expanded by latent state in Python. Spending is kept in float64 without rounding. Annual choice counts are exact sufficient statistics because choice covariates are constant within each household-year.

In [ ]:
def sql_name(name):
    return '"'+name.replace('"','""')+'"'


def prepare_parquet(path, scratch_dir):
    """Validate first, then aggregate; never silently drop malformed records."""
    con=duckdb.connect()
    con.execute(f"SET threads={HMM_NUM_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    scratch_dir=Path(scratch_dir); scratch_dir.mkdir(parents=True,exist_ok=True)
    con.execute("SET temp_directory='"+str(scratch_dir).replace("'","''")+"'")
    con.from_parquet(str(path)).create_view('raw')
    controls=[*GATE_COVARIATE_COLS,*TRANSITION_EVENT_COLUMNS,FRESH_SHARE_2007_COL]
    spends=[s for cfg in CATEGORY_CONFIG.values() for s in cfg['spending']]
    required=['hhid','panel_year','trip_id','purchase_date','food_spend',*controls,*[c+'_y' for c in CATEGORY_NAMES],*spends]
    actual={r[0] for r in con.execute('DESCRIBE raw').fetchall()}
    if set(required)-actual: raise ValueError(f'Missing columns: {set(required)-actual}')
    bad=[f'{sql_name(c)} IS NULL' for c in required]
    numeric=['panel_year','food_spend',*controls,*spends,*[c+'_y' for c in CATEGORY_NAMES]]
    bad += [f'NOT isfinite(CAST({sql_name(c)} AS DOUBLE))' for c in numeric]
    bad += ['panel_year NOT BETWEEN 2008 AND 2017','panel_year != floor(panel_year)',"TRY_CAST(purchase_date AS DATE) IS NULL"]
    bad += [f'{c} NOT IN (0,1)' for c in [*TRANSITION_EVENT_COLUMNS,'K_1']]
    bad += [f'{c}_y NOT IN ({",".join(map(str,cfg["outcomes"]))})' for c,cfg in CATEGORY_CONFIG.items()]
    bad += ['family_size_numeric <= 0','age <= 0','age > 120','income10k_imputed < 0']
    n,b=con.execute('SELECT count(*), count(*) FILTER (WHERE '+ ' OR '.join(bad)+') FROM raw').fetchone()
    if b: raise ValueError(f'{b:,} invalid rows; fix upstream rather than silently filtering.')
    low,high=con.execute(f'SELECT min({FRESH_SHARE_2007_COL}),max({FRESH_SHARE_2007_COL}) FROM raw').fetchone()
    if not n or not (0<low<1) or abs(high-low)>1e-12: raise ValueError('Initial share must be one common value strictly between zero and one.')
    if CHECK_DUPLICATE_TRIPS:
        duplicate=con.execute('SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id HAVING count(*)>1 LIMIT 1').fetchone()
        if duplicate: raise ValueError('Duplicate household-year-trip key.')
    checks=' OR '.join(f'min({c}) != max({c})' for c in controls)
    if con.execute(f'SELECT 1 FROM raw GROUP BY hhid,panel_year HAVING {checks} LIMIT 1').fetchone():
        raise ValueError('Demographics/events/initial share vary within a household-year.')
    con.execute(f'CREATE TEMP TABLE eligible AS SELECT hhid FROM raw GROUP BY hhid HAVING count(DISTINCT panel_year)>={MIN_OBSERVED_YEARS_PER_HOUSEHOLD}')
    con.execute('CREATE VIEW trips AS SELECT r.* FROM raw r SEMI JOIN eligible e USING(hhid)')
    # SQL aliases retain the internal V29 household key to reuse its tested recursion.
    aggs=['CAST(hhid AS VARCHAR) AS household_code','CAST(panel_year AS INTEGER) AS panel_year','count(*) AS n_trips',
          'sum(CAST(food_spend AS DOUBLE)) AS food_total_spend_sum',
          'min(CAST(purchase_date AS DATE)) AS first_purchase_date','max(CAST(purchase_date AS DATE)) AS last_purchase_date']
    aggs += [f'first(CAST({c} AS DOUBLE)) AS {c}' for c in controls]
    for c,cfg in CATEGORY_CONFIG.items():
        aggs += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}_purchase_n',
                 'sum('+ '+'.join(f'CAST({v} AS DOUBLE)' for v in cfg['spending'])+f') AS {c}_spend_sum']
        aggs += [f'sum(CAST({c}_y={k} AS BIGINT)) AS {c}_{v}_n' for k,v in cfg['outcomes'].items() if k]
    hy=con.execute('SELECT '+','.join(aggs)+' FROM trips GROUP BY hhid,panel_year ORDER BY household_code,panel_year').df()
    if hy.empty: raise ValueError('No households pass the observed-year filter.')
    hh=hy.household_code.to_numpy(); starts=np.flatnonzero(np.r_[True,hh[1:]!=hh[:-1]])
    ends=np.r_[starts[1:],len(hy)]
    gaps=(hy.groupby('household_code',sort=False).panel_year.diff()>1)
    if YEAR_GAP_POLICY not in ('observed_step','error'): raise ValueError('Invalid YEAR_GAP_POLICY')
    if gaps.any() and YEAR_GAP_POLICY=='error': raise ValueError(f'{gaps.sum()} observed-year gaps; explicit handling required.')
    hy['chain_step']=hy.groupby('household_code',sort=False).cumcount()+1
    x=np.column_stack([np.ones(len(hy)),hy[TRANSITION_EVENT_COLUMNS].to_numpy(float)])
    mask=np.ones(len(hy),bool); mask[starts]=False
    curr=np.flatnonzero(mask); row=np.full(len(hy),-1,dtype=np.int64); row[curr]=np.arange(len(curr))
    annual_cols=['household_code','panel_year','n_trips','food_total_spend_sum',*TRANSITION_EVENT_COLUMNS]
    annual_cols += [c for c in hy if c.endswith(('_n','_spend_sum')) and c not in annual_cols]
    annual=hy[annual_cols].copy()
    base=np.column_stack([np.ones(len(hy)),hy[GATE_COVARIATE_COLS].to_numpy(float)])
    unique,mapping=np.unique(base,axis=0,return_inverse=True)
    blocks={}
    for c,cfg in CATEGORY_CONFIG.items():
        for gate in cfg['gates']:
            success=hy[f'{c}_{cfg["outcomes"][gate["target_code"]]}_n'].to_numpy(float)
            total=sum(hy[f'{c}_{cfg["outcomes"][v]}_n'].to_numpy(float) for v in gate['eligible_codes'])
            active=np.flatnonzero(total>0)
            if not len(active): raise ValueError(f'No purchases for {gate["key"]}')
            blocks[gate['key']]=GateBlock(gate['key'],c,gate['prefix'],unique,success[active]/total[active],active,int(total.sum()),1.,success[active],total[active]-success[active],mapping[active])
    arrays=ModelArrays({}, {}, {}, np.array([]),np.array([]),{},blocks,np.array([],dtype=int),hy,
        pd.MultiIndex.from_frame(hy[['household_code','panel_year']]),{},hh[starts].tolist(),{},starts,ends,x,curr,row)
    arrays._connection=con
    arrays._annual_statistics=annual
    shell=pd.DataFrame(index=pd.RangeIndex(int(hy.n_trips.sum())))
    shell.attrs['sorting_audit']={'input_trips':n,'retained_trips':len(shell),'minimum_observed_years_required':MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        'trips_removed_by_minimum_years_filter':n-len(shell),'observed_year_gaps':int(gaps.sum()),'year_gap_policy':YEAR_GAP_POLICY}
    print(f'Input {n:,} trips; retained {len(shell):,}; {len(starts):,} households; {len(hy):,} household-years; {gaps.sum():,} gaps.')
    return PreparedData(shell,hy,arrays,annual)


def fit_incidence_models(arrays, progress_bar=None):
    """Six exact logits grouped by food spend; one additional scan caches annual LL."""
    con=arrays._connection
    fields=['CAST(food_spend AS DOUBLE)/10.0 AS spend10','count(*) AS n']
    fields += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}' for c in CATEGORY_NAMES]
    counts=con.execute('SELECT '+','.join(fields)+' FROM trips GROUP BY spend10 ORDER BY spend10').df()
    X=np.column_stack([np.ones(len(counts)),counts.spend10.to_numpy(float)])
    fits={}; aggregates=[]
    for c in CATEGORY_NAMES:
        fits[c]=fit_binomial_count_logit(X,counts[c].to_numpy(float),counts.n.to_numpy(float)-counts[c].to_numpy(float),None,
            [f'{c}_inc_intercept',f'{c}_inc_food_total_spend_10'],f'incidence:{c}')
        a,b=map(float,fits[c]['coef'])
        eta=f'({a!r}+({b!r})*CAST(food_spend AS DOUBLE)/10.0)'
        # Stable log(sigmoid(eta)) / log(1-sigmoid(eta)), no clipping.
        ll=f'(CASE WHEN {c}_y>0 THEN -greatest(-{eta},0) ELSE -greatest({eta},0) END - ln(1+exp(-abs({eta}))))'
        aggregates.append(f'sum({ll}) AS {c}')
    cache=con.execute('SELECT CAST(hhid AS VARCHAR) AS household_code,panel_year,'+','.join(aggregates)+' FROM trips GROUP BY hhid,panel_year ORDER BY household_code,panel_year').df()
    assert np.array_equal(cache.household_code.to_numpy(),arrays.household_year.household_code.to_numpy())
    assert np.array_equal(cache.panel_year.to_numpy(),arrays.household_year.panel_year.to_numpy())
    for c in CATEGORY_NAMES: fits[c]['_annual_loglik']=cache[c].to_numpy(float)
    con.close(); del arrays._connection
    return fits


def initial_gate_weights(prepared):
    """Non-identical emission starts, independent of the constant external pi.

    Smoothed annual fresh share supplies a weak initial ordering. Its only role
    is numerical initialization; subsequent posterior weights come from the HMM.
    """
    hy=prepared.hy
    score=(hy.fv_fresh_n.to_numpy(float)+.5)/(hy.fv_purchase_n.to_numpy(float)+1.)
    z=(score-score.mean())/max(score.std(),1e-8)
    g=expit(.7*np.clip(z,-3,3))
    out=hy[['household_code','panel_year']].copy()
    out['gamma_state_1']=1-g; out['gamma_state_2']=g
    return out


## 3. E-step and safeguarded EM
Forward–backward is vectorized across households with equal sequence lengths. Seven choice regressions use posterior-weighted annual outcome counts. Two transition regressions use at most 64 event patterns. Incidence is separable and fitted once for both models. Initialization uses observed fresh-FV choices only to separate starting emissions; it does not add a likelihood term or alter the fixed initial prior. EM convergence is local, not a global-optimum guarantee.

In [ ]:
def _align_gamma_to_hy(gamma_df: pd.DataFrame, arrays: ModelArrays) -> Tuple[np.ndarray, np.ndarray]:
    """Use zero-copy aligned posterior fast path, retaining a safe keyed fallback."""
    if len(gamma_df) == len(arrays.household_year):
        same_hh = np.array_equal(
            gamma_df["household_code"].astype(str).to_numpy(),
            arrays.household_year["household_code"].astype(str).to_numpy(),
        )
        same_year = np.array_equal(
            gamma_df["panel_year"].to_numpy(dtype=np.int64),
            arrays.household_year["panel_year"].to_numpy(dtype=np.int64),
        )
        if same_hh and same_year:
            return (
                gamma_df["gamma_state_1"].to_numpy(dtype=np.float64, copy=False),
                gamma_df["gamma_state_2"].to_numpy(dtype=np.float64, copy=False),
            )
    gamma_key = pd.MultiIndex.from_frame(gamma_df[["household_code", "panel_year"]])
    aligned = gamma_df.set_index(gamma_key).reindex(arrays.hy_key)
    if aligned[["gamma_state_1", "gamma_state_2"]].isna().any().any():
        raise ValueError("gamma_df cannot be aligned to the household-year index.")
    return aligned["gamma_state_1"].to_numpy(dtype=np.float64), aligned["gamma_state_2"].to_numpy(dtype=np.float64)

def fit_gate_models(
    arrays: ModelArrays,
    gamma_df: pd.DataFrame,
    progress_bar: Optional[Any] = None,
    em_iteration: Optional[int] = None,
    max_em_iters: Optional[int] = None,
    previous_fits: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Fit cached compressed gate likelihoods with one weight pair per eligible row."""
    gamma1, gamma2 = _align_gamma_to_hy(gamma_df, arrays)
    fitted: Dict[str, Any] = {}
    for gate_key in GATE_KEYS:
        block = arrays.gate_blocks[gate_key]
        if progress_bar is not None:
            if em_iteration is None:
                progress_bar.set_description_str(f"Initialization: {progress_bar.n + 1}/{progress_bar.total} | gate={gate_key}")
            else:
                progress_bar.set_description_str(f"EM {em_iteration}/{max_em_iters} | gate={gate_key}")
        # Sum fractional successes/failures over annual groups with the SAME
        # exact design. Four small count vectors replace all trip-state rows.
        n_design = len(block.x_base)
        w1 = gamma1[block.hy_pos]*block.choice_weight
        w2 = gamma2[block.hy_pos]*block.choice_weight
        counts = [np.bincount(block.design_pos, weights=w*count, minlength=n_design)
                  for w,count in ((w1,block.success_count), (w1,block.failure_count),
                                  (w2,block.success_count), (w2,block.failure_count))]
        start = previous_fits[gate_key]["coef"] if previous_fits is not None else None
        fitted[gate_key] = fit_gate_count_logit(
            block.x_base, *counts, start=start,
            param_names=[f"{block.prefix}_{col}" for col in GATE_X_COLS],
            block_name=f"gate:{gate_key}",
        )
        fitted[gate_key]["diagnostics"]["compressed_design_rows"] = n_design
        fitted[gate_key]["diagnostics"]["n_observations"] = block.eligible_rows
        if progress_bar is not None and em_iteration is None:
            progress_bar.update(1)
    return fitted

def log_emission_by_household_year(arrays, incidence_models, gate_params):
    """Evaluate the original trip likelihood using exact annual gate counts."""
    n_hy = len(arrays.household_year)
    common = np.zeros(n_hy)
    for category in CATEGORY_NAMES:
        common += incidence_models[category]["_annual_loglik"]
    state1, state2 = common.copy(), common.copy()
    for key in GATE_KEYS:
        block = arrays.gate_blocks[key]
        coef = np.asarray(gate_params[key]["coef"], dtype=np.float64)
        shared = block.x_base @ np.r_[coef[0], coef[2:]]
        for state_num, destination in ((1.0,state1), (2.0,state2)):
            eta = shared + state_num * coef[1]
            lp, lq = -np.logaddexp(0, -eta), -np.logaddexp(0, eta)
            ll = (block.success_count*lp[block.design_pos]
                  +block.failure_count*lq[block.design_pos])
            # hy_pos is unique in a gate, so direct indexed addition is valid.
            destination[block.hy_pos] += block.choice_weight*ll
    return np.column_stack([state1,state2])

def forward_backward(arrays: ModelArrays, emission_loglik: np.ndarray, pi_state2: float,
                     beta_from1: np.ndarray, beta_from2: np.ndarray):
    """Same log-space recursions, vectorized over equal-length households.

    Observed-year gaps count as one step only under the explicit observed_step policy.
    """
    n_hy = len(arrays.household_year)
    gamma_values = np.empty((n_hy, 2), dtype=np.float64)
    xi_values = np.empty((len(arrays.transition_curr_pos), 4), dtype=np.float64)
    eta1 = arrays.transition_x @ beta_from1
    eta2 = arrays.transition_x @ beta_from2
    log_trans = np.stack([-np.logaddexp(0, eta1), -np.logaddexp(0, -eta1),
                          -np.logaddexp(0, eta2), -np.logaddexp(0, -eta2)], axis=1).reshape(n_hy, 2, 2)
    log_pi = np.array([safe_log(1-pi_state2), safe_log(pi_state2)])
    if not hasattr(arrays, '_sequence_batches'):
        lengths = arrays.hh_ends - arrays.hh_starts
        arrays._sequence_batches = [(np.flatnonzero(lengths == t),
            arrays.hh_starts[lengths == t, None] + np.arange(t)) for t in np.unique(lengths)]
    household_lls = np.empty(len(arrays.hh_starts))
    for hh_indices, pos in arrays._sequence_batches:
        em = emission_loglik[pos]
        trans = log_trans[pos]
        alpha = np.zeros_like(em); backward = np.zeros_like(em)
        alpha[:, 0] = log_pi + em[:, 0]
        for t in range(1, pos.shape[1]):
            alpha[:, t, 0] = em[:, t, 0]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 0], alpha[:, t-1, 1]+trans[:, t, 1, 0])
            alpha[:, t, 1] = em[:, t, 1]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 1], alpha[:, t-1, 1]+trans[:, t, 1, 1])
        ll = np.logaddexp(alpha[:, -1, 0], alpha[:, -1, 1])
        household_lls[hh_indices] = ll
        for t in range(pos.shape[1]-2, -1, -1):
            nxt = em[:, t+1]+backward[:, t+1]
            backward[:, t, 0] = np.logaddexp(trans[:, t+1, 0, 0]+nxt[:, 0], trans[:, t+1, 0, 1]+nxt[:, 1])
            backward[:, t, 1] = np.logaddexp(trans[:, t+1, 1, 0]+nxt[:, 0], trans[:, t+1, 1, 1]+nxt[:, 1])
        gamma = np.exp(alpha + backward - ll[:, None, None])
        gamma /= gamma.sum(axis=2, keepdims=True)
        gamma_values[pos] = gamma
        for t in range(1, pos.shape[1]):
            xi = np.exp(alpha[:, t-1, :, None] + trans[:, t] + em[:, t, None, :]
                        + backward[:, t, None, :] - ll[:, None, None])
            xi /= xi.sum(axis=(1, 2), keepdims=True)
            xi_values[arrays.transition_row_of_hy[pos[:, t]]] = xi.reshape(-1, 4)
    # Sum in original household order to minimize numerical changes.
    total_ll = sum(float(x) for x in household_lls)
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma_values.T
    gamma_df['decoded_state'] = np.where(gamma_values[:, 1] > gamma_values[:, 0], 2, 1)
    curr = arrays.transition_curr_pos
    xi_df = pd.DataFrame({'household_code': arrays.household_year['household_code'].to_numpy()[curr],
        'year_prev': arrays.household_year['panel_year'].to_numpy()[curr-1],
        'year_curr': arrays.household_year['panel_year'].to_numpy()[curr],
        **{c: xi_values[:, j] for j, c in enumerate(['xi_11', 'xi_12', 'xi_21', 'xi_22'])},
        **{c: arrays.transition_x[curr, j] for j, c in enumerate(TRANSITION_EVENT_COLUMNS, 1)}})
    return total_ll, gamma_df, xi_df

def fit_transition_models(
    xi_df: pd.DataFrame,
    start_from1: Optional[np.ndarray] = None,
    start_from2: Optional[np.ndarray] = None,
    arrays: Optional[ModelArrays] = None,
    maxiter: int = 300,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Fit both reference-preserving transition logits from compressed counts."""
    # Six binary flags imply at most 64 distinct designs. Cache their mapping.
    if arrays is not None and hasattr(arrays, "_transition_grouping"):
        X, codes = arrays._transition_grouping
    else:
        flags = np.column_stack([xi_df[c].to_numpy(dtype=np.int8) for c in TRANSITION_EVENT_COLUMNS])
        all_codes = flags @ (1 << np.arange(len(TRANSITION_EVENT_COLUMNS)))
        present, codes = np.unique(all_codes, return_inverse=True)
        X = np.column_stack([np.ones(len(present)),
                             ((present[:, None] >> np.arange(len(TRANSITION_EVENT_COLUMNS))) & 1)])
        codes = codes.astype(np.int32)
        if arrays is not None:
            arrays._transition_grouping = X, codes
    counts = {c:np.bincount(codes, weights=xi_df[c].to_numpy(), minlength=len(X))
              for c in ("xi_11", "xi_12", "xi_21", "xi_22")}
    fit1 = fit_binomial_count_logit(
        X, counts["xi_12"], counts["xi_11"], start_from1,
        [f"trans_from1_{col}" for col in TRANSITION_X_COLS], "transition:from_state_1", maxiter=maxiter,
    )
    fit2 = fit_binomial_count_logit(
        X, counts["xi_22"], counts["xi_21"], start_from2,
        [f"trans_from2_{col}" for col in TRANSITION_X_COLS], "transition:from_state_2", maxiter=maxiter,
    )
    for fit in (fit1,fit2):
        fit["diagnostics"]["compressed_design_rows"] = len(X)
        fit["diagnostics"]["n_observations"] = len(xi_df)
    return fit1, fit2

def _run_monotone_em(prepared, incidence, max_em_iters, tolerance, no_switch=False):
    """Shared EM policy: warm starts, observed-LL guard, persistent plateau.

    A rejected downhill proposal is logged, never relabeled as an improvement.
    Roundoff-sized rejections can count toward a stable plateau only when all
    candidate M-steps satisfy their gradient checks. Larger drops stop with
    converged=False and preserve the last accepted coefficients/posteriors.
    """
    if max_em_iters < 1 or tolerance <= 0:
        raise ValueError('Require max_em_iters >= 1 and tolerance > 0.')
    started_all = time.perf_counter()
    arrays = prepared.arrays
    gates = fit_gate_models(arrays, initial_gate_weights(prepared))
    pi = float(prepared.hy[FRESH_SHARE_2007_COL].iloc[0])
    beta1 = np.zeros(len(TRANSITION_X_COLS)); beta2 = beta1.copy()

    def e_step(g, p, b1, b2):
        emission = log_emission_by_household_year(arrays, incidence, g)
        if no_switch:
            ll, gamma = no_switch_e_step(arrays, emission, p)
            return ll, gamma, None
        return forward_backward(arrays, emission, p, b1, b2)

    ll, gamma, xi = e_step(gates, pi, beta1, beta2)
    trans = []
    if not no_switch:
        # Valid coefficient/covariance containers even if the FIRST EM step
        # is rejected. maxiter=0 evaluates the specified zero coefficients.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            trans = list(fit_transition_models(xi, beta1, beta2, arrays, maxiter=0))
    history = []
    stable = 0
    converged = False
    stop_reason = 'max_iterations_reached'
    label = 'No-switch' if no_switch else 'Proposed'
    incidence_ok = all(incidence[c]['result'].success for c in CATEGORY_NAMES)
    for iteration in range(1, max_em_iters + 1):
        started = time.perf_counter()
        new_pi = pi  # Fixed external initial probability: no M-step update.
        new_trans = [] if no_switch else list(fit_transition_models(xi, beta1, beta2, arrays))
        new_beta1, new_beta2 = (beta1, beta2) if no_switch else tuple(f['coef'] for f in new_trans)
        new_gates = fit_gate_models(arrays, gamma, previous_fits=gates)
        candidate_ll, candidate_gamma, candidate_xi = e_step(new_gates, new_pi, new_beta1, new_beta2)
        candidate_delta = candidate_ll - ll
        failures = sum(not f['result'].success for f in [*new_trans, *new_gates.values()])
        threshold = tolerance + EM_REL_TOL * max(1.0, abs(ll))
        # Tolerance for arithmetic noise, much smaller than an EM stop threshold.
        roundoff = max(1e-8, 64 * np.finfo(float).eps * max(1.0, abs(ll)))
        accepted = bool(np.isfinite(candidate_ll) and candidate_delta >= 0)
        material_drop = not np.isfinite(candidate_ll) or candidate_delta < -roundoff
        delta = 0.0
        if accepted:
            delta = candidate_delta
            pi, gates, trans = new_pi, new_gates, new_trans
            beta1, beta2 = new_beta1, new_beta2
            ll, gamma, xi = candidate_ll, candidate_gamma, candidate_xi
        plateau = not material_drop and delta <= threshold and failures == 0 and incidence_ok
        stable = stable + 1 if plateau else 0
        converged = stable >= EM_STABLE_ITERS
        if material_drop:
            stop_reason = 'downhill_step_rejected'
        elif converged:
            stop_reason = 'stable_likelihood_and_block_gradients'
        history.append({
            'phase': 'EM', 'iteration': iteration, 'log_likelihood': ll,
            'log_likelihood_change': delta, 'candidate_log_likelihood_change': candidate_delta,
            'step_accepted': accepted, 'likelihood_parameter_stage': 'after_M_step',
            'iteration_seconds': time.perf_counter()-started,
            'elapsed_seconds': time.perf_counter()-started_all,
            'converged': converged, 'tolerance': tolerance, 'relative_tolerance': EM_REL_TOL,
            'effective_ll_tolerance': threshold, 'stable_iterations': stable,
            'max_em_iters': max_em_iters, 'optimizer_failures': failures,
            'stop_reason': stop_reason,
        })
        if iteration == 1 or iteration % EM_PRINT_EVERY == 0 or converged or material_drop or iteration == max_em_iters:
            print(f'{label} EM {iteration:03d}/{max_em_iters}: LL={ll:.6f}, '
                  f'delta={delta:.6g}, candidate_delta={candidate_delta:.6g}, '
                  f'block_failures={failures}, stable={stable}/{EM_STABLE_ITERS}')
        if material_drop:
            warnings.warn(f'{label}: downhill EM proposal rejected; last accepted fit retained.', RuntimeWarning)
            break
        if converged:
            break
    return dict(gates=gates, pi=pi, beta1=beta1, beta2=beta2, trans=trans,
                ll=ll, gamma=gamma, history=pd.DataFrame(history), converged=converged,
                stop_reason=stop_reason, estimation_seconds=time.perf_counter()-started_all)

def estimate_model(prepared: PreparedData, max_em_iters: int = MAX_EM_ITERS,
                   tolerance: float = TOL) -> EstimationResult:
    started_all = time.perf_counter()
    print('Fitting incidence models; then safeguarded proposed-model EM...')
    incidence = fit_incidence_models(prepared.arrays)

    out = _run_monotone_em(prepared, incidence, max_em_iters, tolerance)
    for f in [*[incidence[c] for c in CATEGORY_NAMES], *out['trans'], *out['gates'].values()]:
        finalize_covariance(f)
    initial_table = pd.DataFrame()  # Fixed pi has no estimated coefficient.
    trans1, trans2 = out['trans']
    result = EstimationResult(prepared, incidence, out['gates'], out['beta1'], out['beta2'], out['pi'],
        out['gamma'], out['history'], initial_table,
        pd.concat([trans1['table'], trans2['table']], ignore_index=True), out['ll'],
        {'from_state_1': trans1, 'from_state_2': trans2})
    result.optimizer_diagnostics_df = collect_optimizer_diagnostics(result)
    result.estimation_seconds = time.perf_counter()-started_all
    result.converged = out['converged']
    result.stop_reason = out['stop_reason']
    return result

def collect_optimizer_diagnostics(result: EstimationResult) -> pd.DataFrame:
    """Return one diagnostic row for every final optimized logit block."""
    rows = [result.incidence_models[category]["diagnostics"] for category in CATEGORY_NAMES]
    if result.transition_fits is None:
        raise RuntimeError("Final transition fits are unavailable for optimizer diagnostics.")
    rows.extend([
        result.transition_fits["from_state_1"]["diagnostics"],
        result.transition_fits["from_state_2"]["diagnostics"],
    ])
    rows.extend(result.gate_params[key]["diagnostics"] for key in GATE_KEYS)
    return pd.DataFrame(rows)


In [ ]:
@dataclass
class NoSwitchResult:
    """Two latent household classes with exactly zero switching probabilities."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    pi_state2: float
    gamma_df: pd.DataFrame
    log_likelihood: float
    history_df: pd.DataFrame
    converged: bool
    estimation_seconds: float

def no_switch_e_step(arrays, emission_loglik, pi_state2):
    """Exact household mixture likelihood for P = [[1, 0], [0, 1]].

    Sum annual log emissions within each household BEFORE mixing the two
    classes. Each household has one latent class for its entire sequence.
    Direct mixture evaluation avoids clipping structural zeros to epsilon.
    """
    if not 0 < pi_state2 < 1:
        raise ValueError("Initial class probability must be strictly between 0 and 1.")
    household_emission = np.add.reduceat(emission_loglik, arrays.hh_starts, axis=0)
    joint = household_emission + np.array([np.log1p(-pi_state2), np.log(pi_state2)])
    household_ll = np.logaddexp(joint[:, 0], joint[:, 1])
    posterior = np.exp(joint - household_ll[:, None])
    posterior /= posterior.sum(axis=1, keepdims=True)
    # Contiguous annual groups use the identical posterior in every year.
    gamma = np.repeat(posterior, arrays.hh_ends - arrays.hh_starts, axis=0)
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma.T
    gamma_df['decoded_state'] = np.where(gamma[:, 1] > gamma[:, 0], 2, 1)
    return float(household_ll.sum()), gamma_df

def estimate_no_switch_model(proposed, max_em_iters=MAX_EM_ITERS, tolerance=TOL):
    """Re-estimate fixed household classes with the same safeguarded optimizer.

    Reuse the exactly separable incidence fits after raw trip buffers are freed.
    The fixed mixing probability is shared; gate coefficients are independently fit.
    """
    out = _run_monotone_em(proposed.prepared, proposed.incidence_models,
                           max_em_iters, tolerance, no_switch=True)
    # Evaluate only the nested gate Hessians. Incidence covariance is already
    # available from the exactly separable fits shared with the proposed model.
    for fit in out['gates'].values():
        finalize_covariance(fit)
    result = NoSwitchResult(proposed.prepared, proposed.incidence_models, out['gates'],
        out['pi'], out['gamma'], out['ll'], out['history'], out['converged'], out['estimation_seconds'])
    result.stop_reason = out['stop_reason']
    return result


## 4. Complete coefficient tables and posterior profiles
Both models export every estimated coefficient, SE, 95% CI, z, p, and stars. The proposed model also exports incidence, all conditional choices (including all three cereal levels), and category wallet shares. Differences are always State 2 minus State 1. BIC uses the number of independent households. The fixed prior and fixed nested transitions have no significance tests.

In [ ]:
def build_annual_sufficient_statistics(data):
    return data.annual_stats.copy()

def build_transition_matrix(beta_from1: np.ndarray, beta_from2: np.ndarray, x: np.ndarray) -> pd.DataFrame:
    """Build a two-by-two transition matrix at a supplied annual covariate vector."""
    p12 = float(sigmoid(np.dot(beta_from1, x)))
    p22 = float(sigmoid(np.dot(beta_from2, x)))
    return pd.DataFrame({
        "from_state": ["State 1", "State 2"],
        "to_state_1": [1.0 - p12, 1.0 - p22],
        "to_state_2": [p12, p22],
    })

def build_transition_outputs(result: EstimationResult) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build baseline and event-specific transition matrices at mean covariates."""
    hy = result.prepared.hy
    mean_x = np.array([
        1.0 if col == "intercept" else float(hy[col].mean())
        for col in TRANSITION_X_COLS
    ], dtype=np.float64)
    baseline = build_transition_matrix(result.beta_from1, result.beta_from2, mean_x)
    rows: List[pd.DataFrame] = []
    for event in TRANSITION_EVENT_COLUMNS:
        position = TRANSITION_X_COLS.index(event)
        for event_value in [0.0, 1.0]:
            x = mean_x.copy()
            x[position] = event_value
            matrix = build_transition_matrix(result.beta_from1, result.beta_from2, x)
            matrix.insert(0, "event", event)
            matrix.insert(1, "event_value", int(event_value))
            rows.append(matrix)
    return baseline, pd.concat(rows, ignore_index=True)

def _centered_sum_of_squares(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=np.float64)
    total = float(values.sum())
    return max(float(np.dot(values, values)) - total * total / len(values), 0.0)

def infer_posterior_ratio_difference(
    household_stats: pd.DataFrame,
    numerator_stat: str,
    denominator_stat: str,
) -> Dict[str, Any]:
    """Household-clustered delta-method inference for State 2 minus State 1."""
    n_clusters = len(household_stats)
    if n_clusters < 2:
        raise ValueError("At least two households are required for clustered profile inference.")

    numerator_s1 = household_stats[f"{numerator_stat}_s1"].to_numpy(dtype=np.float64, copy=False)
    numerator_s2 = household_stats[f"{numerator_stat}_s2"].to_numpy(dtype=np.float64, copy=False)
    denominator_s1 = household_stats[f"{denominator_stat}_s1"].to_numpy(dtype=np.float64, copy=False)
    denominator_s2 = household_stats[f"{denominator_stat}_s2"].to_numpy(dtype=np.float64, copy=False)
    total_denominator_s1 = float(denominator_s1.sum())
    total_denominator_s2 = float(denominator_s2.sum())

    if total_denominator_s1 <= 0 or total_denominator_s2 <= 0:
        return {
            "state_1_estimate": np.nan,
            "state_1_standard_error": np.nan,
            "state_2_estimate": np.nan,
            "state_2_standard_error": np.nan,
            "difference_state2_minus_state1": np.nan,
            "difference_standard_error": np.nan,
            "z_value": np.nan,
            "p_value": np.nan,
            "significance_stars": "",
            "difference_with_stars": "",
            "inference": "not_available",
            "n_household_clusters": n_clusters,
        }

    state_1_estimate = float(numerator_s1.sum() / total_denominator_s1)
    state_2_estimate = float(numerator_s2.sum() / total_denominator_s2)
    influence_s1 = (numerator_s1 - state_1_estimate * denominator_s1) / total_denominator_s1
    influence_s2 = (numerator_s2 - state_2_estimate * denominator_s2) / total_denominator_s2
    influence_difference = influence_s2 - influence_s1
    finite_sample_correction = n_clusters / (n_clusters - 1.0)
    state_1_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_s1)
    )
    state_2_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_s2)
    )
    difference_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_difference)
    )
    difference = state_2_estimate - state_1_estimate

    if np.isfinite(difference_standard_error) and difference_standard_error > 0:
        z_value = difference / difference_standard_error
        p_value = math.erfc(abs(z_value) / math.sqrt(2.0))
    else:
        z_value = np.nan
        p_value = np.nan
    stars = significance_stars(p_value)

    if not np.isfinite(p_value):
        inference = "not_available"
    elif p_value < 0.10 and difference > 0:
        inference = "state_2_significantly_higher"
    elif p_value < 0.10 and difference < 0:
        inference = "state_1_significantly_higher"
    else:
        inference = "no_statistically_significant_difference"

    return {
        "state_1_estimate": state_1_estimate,
        "state_1_standard_error": state_1_standard_error,
        "state_2_estimate": state_2_estimate,
        "state_2_standard_error": state_2_standard_error,
        "difference_state2_minus_state1": difference,
        "difference_standard_error": difference_standard_error,
        "z_value": z_value,
        "p_value": p_value,
        "significance_stars": stars,
        "difference_with_stars": f"{difference:.6f}{stars}" if np.isfinite(difference) else "",
        "inference": inference,
        "n_household_clusters": n_clusters,
    }

def _collapse_profile_statistics_to_households(
    annual: pd.DataFrame,
    gamma_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Align the estimation sample and collapse posterior-weighted statistics by household."""
    gamma = gamma_df[[
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2"
    ]].copy()
    if gamma.duplicated(["household_code", "panel_year"]).any():
        raise ValueError("Posterior gamma contains duplicate household-year keys.")

    aligned = annual.merge(
        gamma,
        on=["household_code", "panel_year"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    alignment_summary = pd.DataFrame([{
        "panel_household_years_before_alignment": int(len(annual)),
        "panel_household_years_missing_from_posterior": int((aligned["_merge"] == "left_only").sum()),
        "posterior_household_years_missing_from_panel": int((aligned["_merge"] == "right_only").sum()),
        "matched_household_years_retained": int((aligned["_merge"] == "both").sum()),
    }])
    matched = aligned.loc[aligned["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
    if matched.empty:
        raise ValueError("No household-years remain after profile/posterior alignment.")

    excluded = {
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2",
        *TRANSITION_EVENT_COLUMNS,
    }
    stat_columns = [column for column in annual.columns if column not in excluded]
    gamma_state_1 = matched["gamma_state_1"].to_numpy(dtype=np.float64, copy=False)
    gamma_state_2 = matched["gamma_state_2"].to_numpy(dtype=np.float64, copy=False)
    weighted = {"household_code": matched["household_code"]}
    weighted_columns: List[str] = []
    for column in stat_columns:
        values = matched[column].to_numpy(dtype=np.float64, copy=False)
        weighted[f"{column}_s1"] = values * gamma_state_1
        weighted[f"{column}_s2"] = values * gamma_state_2
        weighted_columns.extend([f"{column}_s1", f"{column}_s2"])
    household_stats = (
        pd.DataFrame(weighted)
        .groupby("household_code", sort=False, observed=True)[weighted_columns]
        .sum()
    )
    alignment_summary["matched_households_retained"] = int(len(household_stats))
    return household_stats, alignment_summary

def build_state_profile_summary_with_inference(
    result: EstimationResult,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build six-category profile tables with household-clustered inference."""
    annual = build_annual_sufficient_statistics(result.prepared)
    household_stats, alignment_summary = _collapse_profile_statistics_to_households(
        annual, result.gamma_df
    )
    incidence_rows: List[Dict[str, Any]] = []
    composition_rows: List[Dict[str, Any]] = []
    wallet_rows: List[Dict[str, Any]] = []

    for category, cfg in CATEGORY_CONFIG.items():
        incidence_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_purchase_n", "n_trips"
        )
        incidence_rows.append({
            "profile_section": "purchase_incidence",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "purchase",
            "state_1_purchase_rate": incidence_inference["state_1_estimate"],
            "state_2_purchase_rate": incidence_inference["state_2_estimate"],
            **{key: value for key, value in incidence_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

        for outcome_code, outcome_label in cfg["outcomes"].items():
            if outcome_code == 0:
                continue
            composition_inference = infer_posterior_ratio_difference(
                household_stats,
                f"{category}_{outcome_label}_n",
                f"{category}_purchase_n",
            )
            composition_rows.append({
                "profile_section": "conditional_composition",
                "category": category,
                "outcome_code": outcome_code,
                "outcome_label": outcome_label,
                "state_1_conditional_share": composition_inference["state_1_estimate"],
                "state_2_conditional_share": composition_inference["state_2_estimate"],
                **{key: value for key, value in composition_inference.items()
                   if key not in {"state_1_estimate", "state_2_estimate"}},
            })

        wallet_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_spend_sum", "food_total_spend_sum"
        )
        wallet_rows.append({
            "profile_section": "wallet_share",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "overall_category_wallet_share",
            "state_1_food_wallet_share": wallet_inference["state_1_estimate"],
            "state_2_food_wallet_share": wallet_inference["state_2_estimate"],
            **{key: value for key, value in wallet_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

    incidence = pd.DataFrame(incidence_rows)
    composition = pd.DataFrame(composition_rows)
    wallet = pd.DataFrame(wallet_rows)
    summary = pd.concat([incidence, composition, wallet], ignore_index=True, sort=False)
    preferred_columns = [
        "profile_section", "category", "outcome_code", "outcome_label",
        "state_1_purchase_rate", "state_2_purchase_rate",
        "state_1_conditional_share", "state_2_conditional_share",
        "state_1_food_wallet_share", "state_2_food_wallet_share",
        "state_1_standard_error", "state_2_standard_error",
        "difference_state2_minus_state1", "difference_standard_error",
        "z_value", "p_value", "significance_stars", "difference_with_stars",
        "inference", "n_household_clusters",
    ]
    summary = summary.reindex(columns=preferred_columns)
    valid_stars = {"", "*", "**", "***"}
    observed_stars = set(summary["significance_stars"].fillna("").astype(str).unique())
    if not observed_stars.issubset(valid_stars):
        raise AssertionError(f"Unexpected significance-star values: {sorted(observed_stars - valid_stars)}")
    if len(incidence) != len(CATEGORY_CONFIG) or len(wallet) != len(CATEGORY_CONFIG):
        raise AssertionError("State-profile output does not contain all six categories.")
    expected_composition = sum(len(cfg["outcomes"]) - 1 for cfg in CATEGORY_CONFIG.values())
    if len(composition) != expected_composition:
        raise AssertionError("State-profile output does not contain every positive outcome.")
    return summary, incidence, composition, wallet, alignment_summary


In [ ]:
def fitted_blocks(result, proposed):
    blocks=[(f'{c}_incidence',result.incidence_models[c],False) for c in CATEGORY_NAMES]
    if proposed:
        blocks += [(f'transition_{name}',fit,True) for name,fit in result.transition_fits.items()]
    blocks += [(f'choice_{k}',result.gate_params[k],True) for k in GATE_KEYS]
    return blocks


def coefficient_outputs(result, proposed):
    """Every free parameter exactly once; suppress unidentified Wald inference."""
    frames=[]; covariances=[]
    for name,fit,latent in fitted_blocks(result,proposed):
        t=fit['table'].copy().rename(columns={'std_error':'standard_error','z_value':'z_statistic'})
        t.insert(0,'equation_name',name)
        rank=fit['diagnostics']['hessian_rank']; k=len(fit['coef'])
        valid=rank==k and fit['diagnostics']['covariance_hessian_success']
        cov=fit['cov'].copy()
        if not valid:
            t['standard_error']=np.nan; cov[:]=np.nan
        t.loc[t.standard_error<=0,'standard_error']=np.nan
        t['hessian_based_variance']=t.standard_error**2
        t['z_statistic']=t.estimate/t.standard_error
        t['p_value']=2*norm.sf(abs(t.z_statistic))
        t['significance_stars']=t.p_value.map(significance_stars)
        t['ci_95_lower']=t.estimate-norm.ppf(.975)*t.standard_error
        t['ci_95_upper']=t.estimate+norm.ppf(.975)*t.standard_error
        t['estimate_with_stars']=[f'{v:.6f}{s}' for v,s in zip(t.estimate,t.significance_stars)]
        t['hessian_rank']=rank; t['block_parameter_count']=k
        t['covariance_method']='block M-step Hessian; posterior weights fixed' if latent else 'separable incidence Hessian'
        t['inference_status']='unavailable_rank_or_covariance' if not valid else ('provisional_nonconvergence' if not result.converged else 'approximate' if latent else 'model_based')
        frames.append(t); covariances.append(cov)
    table=pd.concat(frames,ignore_index=True)
    expected=68 if proposed else 54
    assert len(table)==expected and not table.parameter.duplicated().any()
    cov=np.full((expected,expected),np.nan); offset=0
    for block in covariances:
        k=len(block); cov[offset:offset+k,offset:offset+k]=block; offset+=k
    return table,pd.DataFrame(cov,index=table.parameter,columns=table.parameter)


def save_outputs(proposed,nested,destination):
    """Organized machine-readable and reviewable outputs for both fitted models."""
    root=Path(destination); root.mkdir(parents=True,exist_ok=True)
    comparison=[]
    for name,result,is_proposed in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]:
        folder=root/('02_'+name if is_proposed else '03_'+name); folder.mkdir(exist_ok=True)
        coef,cov=coefficient_outputs(result,is_proposed)
        coef.to_csv(folder/'01_full_coefficient_table.csv',index=False)
        cov.to_csv(folder/'02_block_hessian_covariance.csv',index_label='parameter')
        result.gamma_df.rename(columns={'household_code':'hhid'}).to_csv(folder/'06_posterior_household_year.csv',index=False)
        result.history_df.to_csv(folder/'07_estimation_log.csv',index=False)
        diagnostics=pd.DataFrame([f['diagnostics'] for _,f,_ in fitted_blocks(result,is_proposed)])
        diagnostics.to_csv(folder/'09_optimizer_diagnostics.csv',index=False)
        n=len(result.prepared.arrays.hh_starts); ll=result.log_likelihood; k=len(coef)
        row=dict(model=name,log_likelihood=ll,n_parameters=k,AIC=-2*ll+2*k,BIC=-2*ll+k*np.log(n),
            n_households=n,n_household_years=len(result.prepared.hy),n_trips=len(result.prepared.df),
            fixed_initial_pi=result.pi_state2,converged=result.converged,stop_reason=result.stop_reason,
            em_iterations=int(result.history_df.iteration.max()),estimation_seconds=result.estimation_seconds,
            BIC_sample_size_definition='independent households',
            rank_deficient_blocks=int((diagnostics.hessian_rank<diagnostics.parameter_count).sum()))
        comparison.append(row)
        pd.DataFrame([row]).to_csv(folder/'05_model_fit.csv',index=False)
        if is_proposed:
            baseline,events=build_transition_outputs(result)
            baseline['event']='baseline_at_mean_events'
            pd.concat([baseline,events],ignore_index=True).to_csv(folder/'03_transition_matrices_by_event.csv',index=False)
            summary,*_=build_state_profile_summary_with_inference(result)
            summary['inference_method']='household-cluster delta method; posterior weights fixed'
            summary['signed_difference_with_stars']=[f'{v:+.6f}{s}' if np.isfinite(v) else '' for v,s in zip(summary.difference_state2_minus_state1,summary.significance_stars)]
            summary.to_csv(folder/'04_posterior_purchase_profiles.csv',index=False)
            fv=summary.loc[(summary.category=='fv')&(summary.outcome_label=='fresh')].iloc[0]
            label_ok=bool(fv.difference_state2_minus_state1>0)
            pd.DataFrame([dict(state_2_has_higher_fresh_share=label_ok,
                interpretation='Review all six category profiles before labeling states Healthy/Unhealthy')]).to_csv(folder/'10_state_label_diagnostic.csv',index=False)
            if not label_ok: warnings.warn('State 2 has no higher fresh-FV profile. Inspect initialization/local optimum before using Healthy label.')
        else:
            pd.DataFrame(np.eye(2),index=['from_state_1','from_state_2'],columns=['to_state_1','to_state_2']).to_csv(folder/'03_fixed_transition_matrix.csv')
    comp=pd.DataFrame(comparison)
    for m in ['AIC','BIC']: comp['delta_'+m]=comp[m]-comp[m].min()
    provisional=(not comp.converged.all()) or proposed.log_likelihood<nested.log_likelihood-TOL or comp.rank_deficient_blocks.gt(0).any()
    comp['comparison_status']='provisional: inspect convergence, likelihood and Hessians' if provisional else 'EM criteria met; local optima remain possible'
    folder=root/'01_model_comparison'; folder.mkdir(exist_ok=True)
    comp.to_csv(folder/'nested_vs_proposed.csv',index=False)
    pd.DataFrame([proposed.prepared.df.attrs['sorting_audit']]).to_csv(root/'sample_audit.csv',index=False)
    config=dict(input_path=str(INPUT_PATH),run_mode=RUN_MODE,transition_covariates=TRANSITION_EVENT_COLUMNS,
        choice_covariates=GATE_COVARIATE_COLS,fixed_pi=proposed.pi_state2,minimum_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        gap_policy=YEAR_GAP_POLICY,max_em_iters=MAX_EM_ITERS,absolute_tolerance=TOL,relative_tolerance=EM_REL_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__)
    (root/'run_configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    (root/'README.md').write_text('''# V30 results
Both models use the same retained sample and fixed initial prior. Proposed: 68 parameters; nested: 54.
Coefficient tables contain all free parameters, block-Hessian SEs, 95% CIs, z, p and stars.
Stars: * p<.10, ** p<.05, *** p<.01. The fixed initial prior and nested transitions have no SEs.
Latent-block Hessians omit missing-state information; cross-block covariance is NA, not zero.
Posterior profiles are posterior-weighted empirical incidence, conditional composition and category wallet shares.
Profile SEs cluster by household and treat estimated posterior weights as fixed; stars test State 2 minus State 1.
Cereal incidence means classified cereal purchase. Milk code 1 means skimmed; yogurt code 1 means fat-free.
Wallet shares sum all category spending columns, including purchases overridden by outcome priority coding.
BIC uses independent household sequences. The no-switch restriction is on a boundary; no standard chi-square LR p-value is reported.
Read sample_audit.csv for filtering and year gaps. There is no train/test split.
Read estimation logs and state-label diagnostic before interpreting results. Convergence is not proof of a global optimum.
''',encoding='utf-8')
    print(comp[['model','log_likelihood','n_parameters','AIC','BIC','converged']].to_string(index=False))
    return comp


## 5. Execute
Results are written under `V30_Final_HMM_Results`, in separate comparison, proposed, and nested folders. Synthetic mode writes to a separate folder and uses artificial data. Inspect sample audit, convergence, Hessian rank, and state-label diagnostic before using results in the paper. Rerunning replaces files with the same names in this output folder.

In [ ]:
def make_synthetic_parquet(path, n_households=120, seed=SEED):
    """Artificial fixture with all schema fields and both states; never research data."""
    rng=np.random.default_rng(seed); rows=[]; trip=0
    for h in range(n_households):
        state=int(rng.random()<.3)
        for t in range(4):
            events=rng.binomial(1,.15,6)
            if t: state=int(rng.random()<(.8 if state else .2))
            controls=[rng.binomial(1,.5),rng.uniform(1,12),rng.integers(1,6),rng.uniform(25,75)]
            for j in range(12):
                trip+=1
                row=dict(hhid=f'{h:06d}',panel_year=2008+t,trip_id=str(trip),purchase_date=pd.Timestamp(2008+t,1,1)+pd.Timedelta(days=j*20),food_spend=float(rng.integers(10,100)),initial_fresh_fv_share_2007=.3,other_unemployment=0)
                row.update(zip(TRANSITION_EVENT_COLUMNS,events)); row.update(zip(GATE_COVARIATE_COLS,controls))
                for c,cfg in CATEGORY_CONFIG.items():
                    bought=rng.random()<.7
                    y=0
                    if bought:
                        y=1 if rng.random()<(.75 if state else .25) else (2 if c!='cereal' or rng.random()<.5 else 3)
                    row[c+'_y']=y
                    for v in cfg['spending']: row[v]=0.
                    if y: row[cfg['spending'][min(y-1,len(cfg['spending'])-1)]]=2.
                rows.append(row)
    pd.DataFrame(rows).to_parquet(path,index=False)


started=time.perf_counter()
if RUN_MODE not in ('full_estimation','validate_only','synthetic'): raise ValueError('Unknown RUN_MODE')
if RUN_MODE=='synthetic':
    destination=Path.cwd()/'V30_Synthetic_Results'; destination.mkdir(exist_ok=True)
    source=destination/'synthetic_input.parquet'
    make_synthetic_parquet(source)
    iterations=SYNTHETIC_EM_ITERS
else:
    source=INPUT_PATH; destination=OUTPUT_DIR; iterations=MAX_EM_ITERS
    if not source.is_file(): raise FileNotFoundError(source)
prepared=prepare_parquet(source,destination/'duckdb_temp')
print(f'Preparation: {time.perf_counter()-started:.1f} seconds')
if RUN_MODE=='validate_only':
    prepared.arrays._connection.close()
    destination.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([prepared.df.attrs['sorting_audit']]).to_csv(destination/'sample_audit.csv',index=False)
else:
    proposed=estimate_model(prepared,max_em_iters=iterations)
    nested=estimate_no_switch_model(proposed,max_em_iters=iterations)
    comparison=save_outputs(proposed,nested,destination)
    assert proposed.pi_state2==nested.pi_state2==float(prepared.hy[FRESH_SHARE_2007_COL].iloc[0])
    print(f'Total elapsed: {time.perf_counter()-started:.1f} seconds. Results: {destination.resolve()}')
